# ⚾ Homework: Moneyball Part 2, Runs Allowed
In class we found that **on-base percentage** and **slugging percentage** drive **runs scored**. But the A's win games through **run differential** (runs scored − runs allowed), so preventing runs matters just as much.

In this homework you'll build a regression model for **runs allowed**.

Run the setup cells below first. They load the same data and model-report helper we used in class.

## 🧰 Setup: load our tools

In [4]:
import pandas as pd
from pathlib import Path

# statsmodels builds our regression models
import statsmodels.api as sm
from statsmodels.tools.eval_measures import rmse

# sklearn scores our models and splits data into training and testing sets
from sklearn import metrics
from sklearn.model_selection import train_test_split

## 🔎 Load the data and travel back to 2001
Every row is **one team in one season** (for example, the 2001 Oakland A's).

Look at what the code below does:
1. Reads `baseball.csv`.
2. Renames the columns: abbreviations are spelled out using a dictionary, and every label is made lower case with underscores for spaces (`RS` becomes `runs_scored`).
3. Keeps only seasons **before 2002**. This is our "time machine": we pretend we're DePodesta in the 2001–02 off-season.
4. Creates **run differential**: `runs_scored − runs_allowed`. It's stored as `runs_diff`.

In [5]:
# Works whether this notebook sits in the week folder or inside exercise/
data_path = next(p for p in ['data/baseball.csv', '../data/baseball.csv'] if Path(p).exists())
df = pd.read_csv(data_path)

In [6]:
# Spell out the abbreviated column names so the data is easier to read
column_names = {
    'RS': 'runs_scored',
    'RA': 'runs_allowed',
    'W': 'wins',
    'OBP': 'on_base_percentage',
    'SLG': 'slugging_percentage',
    'BA': 'batting_average',
    'G': 'games',
    'OOBP': 'opponent_obp',
    'OSLG': 'opponent_slg',
    'RankSeason': 'rank_season',
    'RankPlayoffs': 'rank_playoffs',
}
df = df.rename(columns=column_names)

# Make every column label lower case (e.g. Team -> team, Year -> year)
df.columns = df.columns.str.lower()

In [ ]:
# Time machine: keep only seasons before Moneyball (2002)
df = df[df['year'] < 2002].reset_index(drop=True)

In [7]:
# Run differential: runs scored minus runs allowed
df['runs_diff'] = df['runs_scored'] - df['runs_allowed']

df.head()

,team,league,year,runs_scored,runs_allowed,wins,on_base_percentage,slugging_percentage,batting_average,playoffs,rank_season,rank_playoffs,games,opponent_obp,opponent_slg,runs_diff
0,ARI,NL,2012,734,688,81,0.328,0.418,0.259,0,NaN,NaN,162,0.317,0.415,46
1,ATL,NL,2012,700,600,94,0.320,0.389,0.247,1,4.0,5.0,162,0.306,0.378,100
2,BAL,AL,2012,712,705,93,0.311,0.417,0.247,1,5.0,4.0,162,0.315,0.403,7
3,BOS,AL,2012,734,806,69,0.315,0.415,0.260,0,NaN,NaN,162,0.331,0.428,-72
4,CHC,NL,2012,613,759,61,0.302,0.378,0.240,0,NaN,NaN,162,0.335,0.424,-146


#### Data Dictionary

| **Column**              | **Original** | **Description**                                          |
| ----------------------- | ------------ | -------------------------------------------------------- |
| **team**                | Team         | Name of the team                                         |
| **league**              | League       | Name of the league                                       |
| **year**                | Year         | Year of the season                                       |
| **runs_scored**         | RS           | Runs scored by the team                                  |
| **runs_allowed**        | RA           | Runs allowed by the team                                 |
| **wins**                | W            | Number of wins                                           |
| **on_base_percentage**  | OBP          | On-Base Percentage                                       |
| **slugging_percentage** | SLG          | Slugging Percentage                                      |
| **batting_average**     | BA           | Batting Average                                          |
| **playoffs**            | Playoffs     | Whether the team made the playoffs (`1 = Yes`, `0 = No`) |
| **rank_season**         | RankSeason   | Regular-season rank among playoff teams (blank if missed playoffs) |
| **rank_playoffs**       | RankPlayoffs | Playoff finish, 1 = won the World Series (blank if missed playoffs) |
| **games**               | G            | Number of games played                                   |
| **opponent_obp**        | OOBP         | Opponent On-Base Percentage                              |
| **opponent_slg**        | OSLG         | Opponent Slugging Percentage                             |
| **runs_diff**           | (new)        | Runs scored minus runs allowed (`runs_scored - runs_allowed`) |


> 💡 **Baseball stats in plain English** (you don't need to be a fan!)
> * **BA (Batting Average)**: hits ÷ at-bats. *How often does a batter get a hit?* This is the stat **traditional scouts loved**.
> * **OBP (On-Base Percentage)**: how often a batter **reaches base by any means** (hit, walk, hit-by-pitch). A walk counts here but not in BA.
> * **SLG (Slugging Percentage)**: total bases ÷ at-bats. Measures **power**: a home run is worth 4, a single is worth 1.
> * **OOBP / OSLG**: the same stats, but for the **opponents' hitters** against our pitchers. Lower is better for us.
>
> 🎯 **The Moneyball bet:** the market paid big money for high **batting average**, but walks (which boost OBP) were nearly free. If OBP matters more than BA for scoring runs, the A's can buy runs cheaply.

## 🛠️ Helper function
**`print_model_report`** is the same helper we used in class. It prints the coefficients and p-values, then R², RMSE, and the average of the dependent variable. By default it scores the model on the data it was trained on. Pass in different `X` and `y` (like a test set) to score it on that data instead.

In [8]:
def print_model_report(model, X=None, y=None):
    # Coefficient and p-value for each variable
    report = pd.DataFrame({'coefficient': model.params, 'p_value': model.pvalues})
    print(report.round(4))
    print()

    # By default, score the model on the data it was trained on
    # (model.model.exog holds the X values; model.model.endog holds the actual y values)
    if X is None:
        X = model.model.exog
        y = model.model.endog

    # How well the model fits
    y_pred = model.predict(X)
    print('R-squared:', round(metrics.r2_score(y, y_pred), 4))
    print('RMSE:', round(rmse(y, y_pred), 2))
    print(f'Average {model.model.endog_names}:', round(y.mean(), 2))

---
## 🧤 Your task: How do we ALLOW fewer runs?
Winning is about **run differential**, so preventing runs counts just as much as scoring them. Let's flip the question around and use the **opponents'** hitting stats against our pitchers:

* **Dependent variable:** `runs_allowed`
* **Try predictors:** `opponent_obp` (opponents' on-base %) and `opponent_slg` (opponents' slugging %)

⚠️ Check for nulls first (`df.info()`): `opponent_obp` and `opponent_slg` are **missing for most of the older seasons**. Drop those rows first (e.g. `df.dropna(subset=['opponent_obp', 'opponent_slg'])`), or the model will error out.

**Your turn:** build and evaluate this model by following the steps below. Then decide: do the coefficients have the sign you'd expect? How does the R² compare with the runs-scored model from class (R² ≈ 0.93 using `on_base_percentage` and `slugging_percentage`)?

### 1. Check for nulls and drop missing rows

In [9]:
# Check for nulls, then drop the rows that are missing opponent_obp or opponent_slg
# Check null counts and column data types
df.info()

# Drop rows where either opponent_obp or opponent_slg is missing
df = df.dropna(subset=['opponent_obp', 'opponent_slg'])

# Verify the missing rows were removed
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1232 entries, 0 to 1231
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   team                 1232 non-null   str    
 1   league               1232 non-null   str    
 2   year                 1232 non-null   int64  
 3   runs_scored          1232 non-null   int64  
 4   runs_allowed         1232 non-null   int64  
 5   wins                 1232 non-null   int64  
 6   on_base_percentage   1232 non-null   float64
 7   slugging_percentage  1232 non-null   float64
 8   batting_average      1232 non-null   float64
 9   playoffs             1232 non-null   int64  
 10  rank_season          244 non-null    float64
 11  rank_playoffs        244 non-null    float64
 12  games                1232 non-null   int64  
 13  opponent_obp         420 non-null    float64
 14  opponent_slg         420 non-null    float64
 15  runs_diff            1232 non-null   int64  
dtyp

### 2. Define X and y
Add a constant to `X` (`sm.add_constant`) so the model can learn an intercept.

In [10]:
# Target variable
y = df['runs_allowed']

# Predictors with an intercept term added
X = sm.add_constant(df[['opponent_obp', 'opponent_slg']])

### 3. Build the model on all the data
Fit the model with `sm.OLS(y, X).fit()`, then print the results with `print_model_report`.

In [ ]:
# Build the model on all the data and print the model report
model = sm.OLS(y, X).fit()

# Print the report using your lab helper function
print_model_report(model)

              coefficient  p_value
const           -877.6051      0.0
opponent_obp    2864.5985      0.0
opponent_slg    1632.7601      0.0

R-squared: 0.9042
RMSE: 27.09
Average runs_allowed: 759.54


### 4. Evaluate on a train/test split
Split the data with `train_test_split(X, y, test_size=0.2, random_state=42)` so everyone gets the same numbers. Fit a model on the training data, then score it on the testing data with `print_model_report(model, X_test, y_test)`.

In [12]:
# Split the data, build the model on the training data, and score it on the testing data
# 1. Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Fit the model only on the training data
model_split = sm.OLS(y_train, X_train).fit()

# 3. Score the model on the test data
print_model_report(model_split, X_test, y_test)

              coefficient  p_value
const           -873.4462      0.0
opponent_obp    2849.1178      0.0
opponent_slg    1636.0656      0.0

R-squared: 0.9128
RMSE: 26.61
Average runs_allowed: 749.23


### 5. Interpret your results
✍️ **Answer here:** Do the coefficients have the sign you'd expect? How does the R² compare with the runs-scored model? How does the testing score compare with the score on all the data?

Yea the signs seem to match for th coeficents because if an opponent scores more slg and is on base more then naturally the runs would increase since thats just the result of that.

R^2 compares with the runs scored model is pretty good at 0.9128 which is greater than 0.90 which means its a good fit.

Testing score is basicaally the same as the score on all the data which means the model predicts it very well and doesnt overfit,